In [0]:
from pyspark.sql import functions as F

cust_schema = """
    customer_id STRING,
    email STRING,
    first_name STRING,
    last_name STRING,
    gender STRING,
    street STRING,
    city STRING,
    country_code STRING,
    row_status STRING,
    row_time TIMESTAMP
"""

In [0]:
def process_delete_requests():
    deltes_df = (
        spark.readStream
            .table('dev.bookstore_bronze.bookstore_bronze')
            .filter("topic = 'customers'")
            .select(F.from_json(F.col("value").cast('string'), schema=cust_schema).alias("data"))
            .select("data.*", F.col('data.row_time').alias("request_timestamp"))
            .filter(F.col("row_status") == "delete")
            .select(
                "customer_id", 
                "request_timestamp", 
                F.date_add("request_timestamp", 30).alias("deadline"),
                F.lit("requested").alias("status")
            )
            .writeStream
               .outputMode("append")
               .option("checkpointLocation", "dbfs:/Volumes/dev/landing_zone/kafka_source/checkpoints/delete_requestes")
               .trigger(availableNow=True)
               .table("dev.silver.delete_requests")
    )


In [0]:
dbutils.fs.rm("dbfs:/Volumes/dev/landing_zone/kafka_source/checkpoints/delete_requestes", True)

In [0]:
process_delete_requests()

In [0]:
%sql
-- Delete customers_silver where delete request is older than 30 days
DELETE FROM dev.silver.customers_silver
WHERE customer_id in (
  SELECT 
    customer_id
  FROM dev.silver.delete_requests
  WHERE status = 'requested'
)

In [0]:
%sql
show tables in dev.silver;

In [0]:
%sql
select * from dev.silver.delete_requests

In [0]:
%sql
    
select
  key,
  cast(value as string):row_status as row_status,
  topic,
  timestamp
from dev.bookstore_bronze.bookstore_bronze 
where topic = 'customers' and cast(value as string):row_status = 'delete'

In [0]:
%sql
select 
  * 
from table_changes('dev.silver.customers_silver', 2)
where _change_type = 'delete'

In [0]:
def process_deletes(microBatchDF, batchId):
    
    microBatchDF.filter("_change_type = 'delete'")\
        .createOrReplaceTempView("deletes")
    
    microBatchDF.sparkSession.sql(
        """
        DELETE FROM dev.silver.customers_orders
        WHERE customer_id in (
            SELECT customer_id FROM deletes
        )    
        """
    )

    microBatchDF.sparkSession.sql(
        """
            MERGE INTO dev.silver.delete_requests r
            USING deletes d
            ON r.customer_id = d.customer_id
            WHEN MATCHED THEN 
                UPDATE SET 
                    r.status = 'deleted'
        """
    )


In [0]:
def process_propagating_deletes():
    deleteDF = (
        spark.readStream
            .format("delta")
            .option("readChangeData", 'true')
            .option("startingVersion", 2)
            .table("dev.silver.customers_silver")
    )

    deleteDF.writeStream\
        .foreachBatch(process_deletes)\
        .option("checkpointLocation", "dbfs:/Volumes/dev/landing_zone/kafka_source/checkpointsdeletes")\
        .trigger(availableNow=True)\
        .start()

In [0]:
dbutils.fs.rm("dbfs:/Volumes/dev/landing_zone/kafka_source/checkpoints/deletes_display", True)

deleteDF = (
        spark.readStream
            .format("delta")
            .option("readChangeData", 'true')
            .option("startingVersion", 2)
            .table("dev.silver.customers_silver").select("customer_id", "_change_type").filter("_change_type = 'delete'")
    )
display(deleteDF, checkpointLocation="dbfs:/Volumes/dev/landing_zone/kafka_source/checkpoints/deletes_display")


In [0]:
%sql
select * from dev.silver.customers_orders

In [0]:
%sql
DESC HISTORY dev.silver.customers_orders

In [0]:
%sql
-- get the difference between two versions
SELECT * FROM dev.silver.customers_orders@v1
EXCEPT
SELECT * FROM dev.silver.customers_orders

In [0]:
for i in range(10):
    dbutils.fs.rm(f"dbfs:/Volumes/dev/landing_zone/kafka_source/book_store/{str(i).zfill(2)}")
